In [82]:
import pandas as pd
import numpy as np
import pvlib
import matplotlib.pyplot as plt
import seaborn as sns

### Load the data and setting up the index

In [83]:
df = pd.read_csv("../../Data/solar2022.csv",skiprows=2)

In [84]:
df['datetime']=pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])

In [85]:
df = df.drop(columns=['Year', 'Month', 'Day', 'Hour', 'Minute'])

In [86]:
df = df.set_index('datetime').sort_index()

In [87]:
df.index = df.index.tz_localize('Etc/GMT-1')

### Define the site location

In [88]:
location = pvlib.location.Location(
    latitude=40.97,
    longitude=-4.54,
    altitude=895,
    tz='UTC'
)

## Time-based features

### Compute solar position

In [89]:
solar_pos = location.get_solarposition(df.index)

In [90]:
# Solar zenith angle (refraction-corrected)
df['solar_zenith'] = solar_pos['apparent_zenith']

In [91]:
# Solar elevation (refraction-corrected)
df['solar_elevation'] = solar_pos['apparent_elevation']

In [92]:
# Extraterrestrial irradiance on horizontal surface
dni_extra = pvlib.irradiance.get_extra_radiation(df.index)
cos_zenith = pvlib.tools.cosd(df['solar_zenith'])
df['ET_irradiance'] = (dni_extra * cos_zenith).clip(lower=0)

In [93]:
mae = (df['Solar Zenith Angle'] - df['solar_zenith']).abs().mean()
print(f'Zenith MAE vs NSRDB: {mae:.3f}°')  # expect < 0.5°

Zenith MAE vs NSRDB: 0.003°


###  Hour of day (cyclical encoding)

In [94]:
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

### Day of year (seasonality encoding)

In [95]:
df['doy_sin'] = np.sin(2 * np.pi * df.index.day_of_year / 365)
df['doy_cos'] = np.cos(2 * np.pi * df.index.day_of_year / 365)

### Calendar features

In [96]:
df['month']        = df.index.month
df['week_of_year'] = df.index.isocalendar().week.astype(int)
df['day_of_week']  = df.index.dayofweek

### Validate

In [98]:
print(df[['hour_sin','hour_cos','doy_sin','doy_cos','month','week_of_year','day_of_week']].head(10))

                           hour_sin      hour_cos   doy_sin   doy_cos  month  \
datetime                                                                       
2022-01-01 00:30:00+01:00  0.000000  1.000000e+00  0.017213  0.999852      1   
2022-01-01 01:30:00+01:00  0.258819  9.659258e-01  0.017213  0.999852      1   
2022-01-01 02:30:00+01:00  0.500000  8.660254e-01  0.017213  0.999852      1   
2022-01-01 03:30:00+01:00  0.707107  7.071068e-01  0.017213  0.999852      1   
2022-01-01 04:30:00+01:00  0.866025  5.000000e-01  0.017213  0.999852      1   
2022-01-01 05:30:00+01:00  0.965926  2.588190e-01  0.017213  0.999852      1   
2022-01-01 06:30:00+01:00  1.000000  6.123234e-17  0.017213  0.999852      1   
2022-01-01 07:30:00+01:00  0.965926 -2.588190e-01  0.017213  0.999852      1   
2022-01-01 08:30:00+01:00  0.866025 -5.000000e-01  0.017213  0.999852      1   
2022-01-01 09:30:00+01:00  0.707107 -7.071068e-01  0.017213  0.999852      1   

                           week_of_year

In [97]:
df.head()

,Temperature,Clearsky DHI,Clearsky DNI,Clearsky GHI,Cloud Type,Relative Humidity,Pressure,Wind Direction,Wind Speed,DHI,...,solar_zenith,solar_elevation,ET_irradiance,hour_sin,hour_cos,doy_sin,doy_cos,month,week_of_year,day_of_week
datetime,,,,,,,,,,,,,,,,,,,,,
2022-01-01 00:30:00+01:00,4.9,0,0,0,1,69.27,914,172,3.4,0,...,159.051591,-69.051591,0.0,0.000000,1.000000,0.017213,0.999852,1,52,5
2022-01-01 01:30:00+01:00,5.2,0,0,0,4,57.26,918,183,1.6,0,...,161.960001,-71.960001,0.0,0.258819,0.965926,0.017213,0.999852,1,52,5
2022-01-01 02:30:00+01:00,5.3,0,0,0,0,55.49,919,184,1.5,0,...,156.997377,-66.997377,0.0,0.500000,0.866025,0.017213,0.999852,1,52,5
2022-01-01 03:30:00+01:00,5.3,0,0,0,0,54.39,919,184,1.5,0,...,147.661657,-57.661657,0.0,0.707107,0.707107,0.017213,0.999852,1,52,5
2022-01-01 04:30:00+01:00,5.3,0,0,0,4,53.39,919,184,1.4,0,...,136.822203,-46.822203,0.0,0.866025,0.500000,0.017213,0.999852,1,52,5


## Lagged Weather Features

### GHI lags

In [99]:
df['GHI_lag_1h']  = df['GHI'].shift(1)
df['GHI_lag_3h']  = df['GHI'].shift(3)
df['GHI_lag_24h'] = df['GHI'].shift(24)

### Temperaturen lags

In [100]:
df['temp_lag_1h']  = df['Temperature'].shift(1)
df['temp_lag_3h']  = df['Temperature'].shift(3)
df['temp_lag_24h'] = df['Temperature'].shift(24)

### Humidity lags

In [101]:
df['humidity_lag_1h']  = df['Relative Humidity'].shift(1)
df['humidity_lag_3h']  = df['Relative Humidity'].shift(3)
df['humidity_lag_24h'] = df['Relative Humidity'].shift(24)

### Validate

print(df[['GHI', 'GHI_lag_1h', 'GHI_lag_3h', 'GHI_lag_24h']].head(30))

## Rolling statistics

### Rolling mean 

In [104]:
df['GHI_rollmean_3h']  = df['GHI'].shift(1).rolling(window=3).mean()
df['GHI_rollmean_6h']  = df['GHI'].shift(1).rolling(window=6).mean()
df['GHI_rollmean_24h'] = df['GHI'].shift(1).rolling(window=24).mean()

###  Rolling standard deviation

In [105]:
df['GHI_rollstd_3h']  = df['GHI'].shift(1).rolling(window=3).std()
df['GHI_rollstd_6h']  = df['GHI'].shift(1).rolling(window=6).std()
df['GHI_rollstd_24h'] = df['GHI'].shift(1).rolling(window=24).std()

### Validate

In [106]:
print(df[['GHI','GHI_rollmean_3h','GHI_rollmean_6h','GHI_rollmean_24h',
          'GHI_rollstd_3h','GHI_rollstd_6h','GHI_rollstd_24h']].iloc[20:35])

                           GHI  GHI_rollmean_3h  GHI_rollmean_6h  \
datetime                                                           
2022-01-01 20:30:00+01:00    0         7.666667       138.833333   
2022-01-01 21:30:00+01:00    0         0.000000        93.500000   
2022-01-01 22:30:00+01:00    0         0.000000        37.666667   
2022-01-01 23:30:00+01:00    0         0.000000         3.833333   
2022-01-02 00:30:00+01:00    0         0.000000         0.000000   
2022-01-02 01:30:00+01:00    0         0.000000         0.000000   
2022-01-02 02:30:00+01:00    0         0.000000         0.000000   
2022-01-02 03:30:00+01:00    0         0.000000         0.000000   
2022-01-02 04:30:00+01:00    0         0.000000         0.000000   
2022-01-02 05:30:00+01:00    0         0.000000         0.000000   
2022-01-02 06:30:00+01:00    0         0.000000         0.000000   
2022-01-02 07:30:00+01:00    0         0.000000         0.000000   
2022-01-02 08:30:00+01:00    0         0.000000 

## Feature interactions

## Cloud cover Proxy

In [107]:
df['cloud_cover_proxy'] = (df['Clearsky GHI'] - df['GHI']).clip(lower=0) / (df['Clearsky GHI'] + 1)

### Cloud cover × solar elevation

In [111]:
df['cloud_x_elevation'] = df['cloud_cover_proxy'] * df['solar_elevation'].clip(lower=0)

### Humidity × temperature

In [112]:
df['humidity_x_temp'] = df['Relative Humidity'] * df['Temperature']

In [113]:
print(df[['GHI', 'Clearsky GHI', 'cloud_cover_proxy', 
          'cloud_x_elevation', 'humidity_x_temp']].iloc[9:18])

                           GHI  Clearsky GHI  cloud_cover_proxy  \
datetime                                                          
2022-01-01 09:30:00+01:00   18            96           0.804124   
2022-01-01 10:30:00+01:00  141           245           0.422764   
2022-01-01 11:30:00+01:00  219           369           0.405405   
2022-01-01 12:30:00+01:00  296           446           0.335570   
2022-01-01 13:30:00+01:00  464           464           0.000000   
2022-01-01 14:30:00+01:00  272           428           0.363636   
2022-01-01 15:30:00+01:00  335           335           0.000000   
2022-01-01 16:30:00+01:00  203           203           0.000000   
2022-01-01 17:30:00+01:00   23            55           0.571429   

                           cloud_x_elevation  humidity_x_temp  
datetime                                                       
2022-01-01 09:30:00+01:00           5.318710          368.753  
2022-01-01 10:30:00+01:00           6.219015          569.633  
2022-0

In [117]:
df.to_csv('../../Outputs/df_engineered.csv')
print(f'Final dataset shape: {df.shape}')
print(f'Total features: {df.shape[1]}')

Final dataset shape: (8760, 48)
Total features: 48


In [118]:
df_clean = df.dropna()

In [119]:
print(f'Rows before dropna: {len(df)}')
print(f'Rows after dropna:  {len(df_clean)}')
print(f'Rows dropped:       {len(df) - len(df_clean)}')

Rows before dropna: 8760
Rows after dropna:  8736
Rows dropped:       24


In [120]:
# Save the clean version
df_clean.to_csv('../../Outputs/df_engineered.csv')
print('Saved successfully.')

Saved successfully.
